**ANALYSIS OF 23daVinci/CFPB_Complaints dataset**


`Braeakdown`: The dataset is in the form of categories and complaint narrative

Some rows have a categorie section with multiple categories seperated by commas

In [1]:
from datasets import load_dataset


dataset = load_dataset("23daVinci/CFPB_Complaints")
print(dataset)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['Product', 'Consumer complaint narrative'],
        num_rows: 2240280
    })
    validation: Dataset({
        features: ['Product', 'Consumer complaint narrative'],
        num_rows: 560071
    })
    test: Dataset({
        features: ['Product', 'Consumer complaint narrative'],
        num_rows: 28287
    })
})
{'Product': 'Credit reporting, credit repair services, or other personal consumer reports', 'Consumer complaint narrative': "On ( XX/XX/2023 ) I sent a letter regarding inaccurate Late payments, inaccurate negative accounts, bankruptcies and unknown things on my credit report. To this day over 60 days later I have not received a response yet. feel like i'm being taken advantage of and being ignored of my disputes. Section 611 ( a ) it is plainly stated that failure to investigate these items within 30 days gives a reason to immediately remove those negative items from my credit report as well as correct my late payments as 

**Class distribution**



Classes with considerably low counts: 
    -`Debt or credit management`
    -`Prepaid card`
    -`Bank account or service`
    -`Consumer loan`
    -`Payday loan`
    -`title loan`
    -`or personal loan`
    -`Money transfers`
    -`personal loan`
    -`or advance loan`
    -`Other financial service`
    -`Virtual currency`


In [2]:
from collections import defaultdict
def split_categories(txt) -> list[str]:
    return txt.split(",")



categories_dict = defaultdict(int)

for item in dataset["train"]:
    categories = split_categories(item["Product"])
    for category in categories:
        categories_dict[category] += 1

print(categories_dict)








defaultdict(<class 'int'>, {'Credit reporting': 664385, ' credit repair services': 639368, ' or other personal consumer reports': 639368, 'Credit reporting or other personal consumer reports': 768491, 'Student loan': 40110, 'Mortgage': 101833, 'Money transfer': 74661, ' virtual currency': 74661, ' or money service': 74661, 'Debt collection': 256301, 'Vehicle loan or lease': 30370, 'Credit card': 60419, 'Checking or savings account': 107593, 'Credit card or prepaid card': 86065, 'Debt or credit management': 2093, 'Prepaid card': 5844, 'Bank account or service': 11789, 'Consumer Loan': 7493, 'Payday loan': 21403, ' title loan': 20021, ' or personal loan': 13653, 'Money transfers': 1186, ' personal loan': 6368, ' or advance loan': 6368, 'Other financial service': 231, 'Virtual currency': 13})


In [3]:
print(len(categories_dict))

26


Merging done based on observed counts. This is done to ensure workable categorization

- Credit reporting
- Debt collection
- Mortgage
- Student loan
- Credit card / prepaid card
- Banking
- Money transfer / virtual currency
- Vehicle loan / lease
- Payday / personal / consumer loan
- Debt or credit management
- Other financial service

In [6]:
def normalize_product(product):
    if product is None:
        return None

    product = product.strip()

    credit_reporting = {
        "Credit reporting",
        "Credit reporting, credit repair services, or other personal consumer reports",
        "Credit reporting or other personal consumer reports",
    }

    credit_card = {
        "Credit card",
        "Credit card or prepaid card",
        "Prepaid card",
    }

    banking = {
        "Bank account or service",
        "Checking or savings account",
    }

    money_transfer = {
        "Money transfer, virtual currency, or money service",
        "Money transfers",
        "Virtual currency",
    }

    personal_loans = {
        "Consumer Loan",
        "Payday loan",
        "Payday loan, title loan, or personal loan",
        "Personal loan",
        "Title loan",
    }

    if product in credit_reporting:
        return "Credit reporting"

    if product in credit_card:
        return "Credit card / prepaid card"

    if product in banking:
        return "Banking"

    if product in money_transfer:
        return "Money transfer / virtual currency"

    if product in personal_loans:
        return "Payday / personal / consumer loan"

    if product == "Vehicle loan or lease":
        return "Vehicle loan / lease"

    if product == "Debt collection":
        return "Debt collection"

    if product == "Mortgage":
        return "Mortgage"

    if product == "Student loan":
        return "Student loan"

    if product == "Debt or credit management":
        return "Debt or credit management"

    if product == "Other financial service":
        return "Other financial service"

    return "Other financial service"


In [7]:
def add_clean_product(example):
    example["clean_product"] = normalize_product(example["Product"])
    return example
train_ds = dataset["train"]
train_ds= train_ds.map(add_clean_product)

Map:   0%|          | 0/2240280 [00:00<?, ? examples/s]

In [8]:
from collections import Counter

counts = Counter(train_ds["clean_product"])

for label, count in counts.most_common():
    print(label, count)

Credit reporting 1432876
Debt collection 256301
Credit card / prepaid card 152328
Banking 119382
Mortgage 101833
Money transfer / virtual currency 75860
Student loan 40110
Vehicle loan / lease 30370
Payday / personal / consumer loan 22528
Other financial service 6599
Debt or credit management 2093


**WORKING WITH ONLY A SMALL SAMPLE OF EXAMPLES**

30,000 samples will first be used. Stratified splits will be performed


11 classes so roughly each class should have  approx-> 2727

In [9]:
train_ds = train_ds.class_encode_column("clean_product")

label_names = train_ds.features["clean_product"].names

print(label_names)


Casting to class labels:   0%|          | 0/2240280 [00:00<?, ? examples/s]

['Banking', 'Credit card / prepaid card', 'Credit reporting', 'Debt collection', 'Debt or credit management', 'Money transfer / virtual currency', 'Mortgage', 'Other financial service', 'Payday / personal / consumer loan', 'Student loan', 'Vehicle loan / lease']


In [10]:
import numpy as np
from collections import Counter

def make_balanced_subset(ds, label_col="clean_product", n_total=30_000, seed=42):
    rng = np.random.default_rng(seed)

    labels = np.array(ds[label_col])
    unique_labels = np.unique(labels)

    samples_per_class = n_total // len(unique_labels)

    selected_indices = []

    for label in unique_labels:
        class_indices = np.flatnonzero(labels == label)
        n_take = min(samples_per_class, len(class_indices))

        chosen = rng.choice(
            class_indices,
            size=n_take,
            replace=False
        )

        selected_indices.extend(chosen.tolist())

    subset = ds.select(selected_indices).shuffle(seed=seed)
    return subset

train_subset = make_balanced_subset(
    train_ds,
    label_col="clean_product",
    n_total=30_000
)

counts = Counter(train_subset["clean_product"])

for label_id, count in sorted(counts.items()):
    print(label_names[label_id], count)

Banking 2727
Credit card / prepaid card 2727
Credit reporting 2727
Debt collection 2727
Debt or credit management 2093
Money transfer / virtual currency 2727
Mortgage 2727
Other financial service 2727
Payday / personal / consumer loan 2727
Student loan 2727
Vehicle loan / lease 2727


**Perform encodings and get subset for train,validation, and test

In [ ]:
validation_set = dataset["validation"].map(add_clean_product)
test_set = dataset["test"].map(add_clean_product)

validation_subset = make_balanced_subset(
    validation_set,
    label_col="clean_product",
    n_total=10_000
)
test_subset = make_balanced_subset(
    test_set,
    label_col="clean_product",
    n_total=20_000
)


